In [ ]:
! pip install transformers datasets torch

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from datasets import Dataset
import torch

In [4]:
# import dataset
dataset = pd.read_csv('../dataset/cleaned_dataset_bert.csv')
dataset.shape

(51093, 2)

In [ ]:
dataset.head()

In [10]:
dataset['status'].value_counts()

status
Normal                  16040
Depression              15094
Suicidal                10644
Anxiety                  3623
Bipolar                  2501
Stress                   2296
Personality disorder      895
Name: count, dtype: int64

In [11]:
# configuration
MODEL_NAME = "bert-base-uncased"
NUM_LABELS = 7
MAX_LEN = 512
BATCH_SIZE = 8   
EPOCHS = 3      
LEARNING_RATE = 2e-5 

In [42]:
# label encoding
unique_labels = sorted(dataset['status'].unique())
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}
NUM_LABELS = len(unique_labels) # Dynamically set NUM_LABELS

dataset['label'] = dataset['status'].map(label_to_id)
print(f"Unique statuses:  {unique_labels}")
print(f"Label mapping: {label_to_id}")

print("-"*20)

Unique statuses:  ['Anxiety', 'Bipolar', 'Depression', 'Normal', 'Personality disorder', 'Stress', 'Suicidal']
Label mapping: {'Anxiety': 0, 'Bipolar': 1, 'Depression': 2, 'Normal': 3, 'Personality disorder': 4, 'Stress': 5, 'Suicidal': 6}
--------------------


In [43]:
dataset.head()

,processed_text,status,label
0,oh my gosh,Anxiety,0
1,trouble sleeping confused mind restless heart ...,Anxiety,0
2,all wrong back off dear forward doubt stay in ...,Anxiety,0
3,i have shifted my focus to something else but ...,Anxiety,0
4,i am restless and restless it is been a month ...,Anxiety,0


In [ ]:
X = dataset['processed_text'].values
y = dataset['status'].values

y_encoded = dataset['label'].map(label_to_id)
print(f"Unique Classes: {unique_labels}")
print(f"Label Mapping: {label_to_id}")
print("-" * 40)

Unique Classes: ['Anxiety', 'Bipolar', 'Depression', 'Normal', 'Personality disorder', 'Stress', 'Suicidal']
Label Mapping: {'Anxiety': 0, 'Bipolar': 1, 'Depression': 2, 'Normal': 3, 'Personality disorder': 4, 'Stress': 5, 'Suicidal': 6}
----------------------------------------


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [50]:
# --- INSTALL NECESSARY LIBRARIES ---
# !pip install transformers datasets pandas numpy torch scikit-learn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

# --- CONFIGURATION ---
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5


# --- 1. DATA PREPARATION (REPLACE WITH YOUR ACTUAL DATA) ---
""" data = {
    'Statement': [
        "I feel terrible, there is no hope left for me.",
        "I'm feeling good, just planning a new project.",
        "I can't sleep and I'm restless all the time.",
        "This is an intense low, but I will manage.",
        "I am fine, everything is normal today.",
        "The mood swings are extreme, up and down.",
        "Dark thoughts are overwhelming my mind.",
    ] * 500, # Increased size to better represent your 51k samples
    'Status': [
        'Suicidal', 'Normal', 'Anxiety', 'Depression', 'Normal', 'Bipolar', 'Suicidal',
    ] * 500
}
df = pd.DataFrame(data) """

df = pd.read_csv('../dataset/cleaned_dataset_bert.csv')
df.head()

,processed_text,status
0,oh my gosh,Anxiety
1,trouble sleeping confused mind restless heart ...,Anxiety
2,all wrong back off dear forward doubt stay in ...,Anxiety
3,i have shifted my focus to something else but ...,Anxiety
4,i am restless and restless it is been a month ...,Anxiety


In [ ]:


# Separate X (features) and y (labels)
X = df['processed_text']
y = df['status']

# --- 2. LABEL ENCODING ---
unique_labels = sorted(y.unique())
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}
NUM_LABELS = len(unique_labels) # Dynamically set NUM_LABELS

y_encoded = y.map(label_to_id)
print(f"Unique Classes: {unique_labels}")
print(f"Label Mapping: {label_to_id}")
print("-" * 40)

# --- 3. TRAIN-TEST SPLIT (Corrected Logic) ---
# Split X (statements) and y_encoded (numerical labels). Stratify ensures balanced splits.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded # Stratify using the numerical labels
)

# Re-assemble into Hugging Face-compatible DataFrames
train_df = pd.DataFrame({'processed_text': X_train, 'label': y_train})
test_df = pd.DataFrame({'processed_text': X_test, 'label': y_test})

# Convert Pandas DataFrames to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
print("Train/Test Split and Dataset Conversion Complete.")
print("-" * 40)

# --- 4. TOKENIZATION ---
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['processed_text'], 
        padding='max_length', 
        truncation=True, 
        max_length=MAX_LEN
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Rename the 'label' column to 'labels' as required by BertForSequenceClassification
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

# Remove original text column
tokenized_train = tokenized_train.remove_columns(["processed_text"])
tokenized_test = tokenized_test.remove_columns(["processed_text"])

# Set format for PyTorch
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")
print("Tokenization complete.")
print("-" * 40)


# --- 5. MODEL LOADING AND METRICS ---
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id_to_label,
    label2id=label_to_id
)

def compute_metrics(p):
    predictions = np.argmax(p.predictions, axis=1)
    # Use 'macro' average to give equal weight to each class (good for imbalance)
    return {
        'accuracy': accuracy_score(p.label_ids, predictions),
        'f1_macro': classification_report(p.label_ids, predictions, output_dict=True, zero_division=0)['macro avg']['f1-score'],
        'recall_macro': classification_report(p.label_ids, predictions, output_dict=True, zero_division=0)['macro avg']['recall'],
    }


# --- 6. TRAINING ARGUMENTS AND TRAINER SETUP ---
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    logging_steps=100,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available() 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# --- 7. TRAIN THE MODEL ---
print("Starting BERT Fine-Tuning... (Requires GPU)")
trainer.train()

# --- 8. FINAL EVALUATION ---
print("\n--- Final Test Set Evaluation ---")
test_predictions = trainer.predict(tokenized_test)
final_predictions = np.argmax(test_predictions.predictions, axis=1)
final_labels = test_predictions.label_ids

print("\nDetailed Multi-Class Report (BERT):")
print(classification_report(final_labels, final_predictions, target_names=unique_labels, zero_division=0))

c:\Users\Yohan\anaconda3\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Yohan\anaconda3\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
W1112 15:00:42.993000 21040 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\Yohan\anaconda3\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Unique Classes: ['Anxiety', 'Bipolar', 'Depression', 'Normal', 'Suicidal']
Label Mapping: {'Anxiety': 0, 'Bipolar': 1, 'Depression': 2, 'Normal': 3, 'Suicidal': 4}
----------------------------------------
Train/Test Split and Dataset Conversion Complete.
----------------------------------------


c:\Users\Yohan\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting BERT Fine-Tuning... (Requires GPU)


  0%|          | 0/1050 [00:00<?, ?it/s]

c:\Users\Yohan\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


KeyboardInterrupt: 

In [ ]:
# convert pandas dataframe to hugging face dataset objects
train_dataset = Dataset.from_pandas(train_df[['processed_text', 'label']], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[['processed_text', 'label']], preserve_index=False)